In [ ]:
# @title Setup
from google.cloud import bigquery
from google.colab import data_table
import bigframes.pandas as bpd

project = 'smiling-rhythm-486213-b9'
location = 'US'
client = bigquery.Client(project=project, location=location)
data_table.enable_dataframe_formatter()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
from datetime import datetime
current_year = datetime.now().year
current_year

In [ ]:
def query_to_dataframe(query: str) -> pd.DataFrame:
    """
    Executes a SQL query in BigQuery and returns a Pandas DataFrame.
    """
    try:
        df = client.query(query).to_dataframe()
        print(f"Query executed successfully. Retrieved {df.shape[0]} rows.")
        return df
    except Exception as e:
        print(f"Error executing query: {e}")
        return pd.DataFrame()

## IV/ Competitive Insights & Operational Efficiency

### Question 12: Which boroughs or zones have the highest and lowest trip volumes, and how do they compare over time?


In [ ]:
query_trip_volume_by_borough = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.trip_volume_by_borough`
"""
trip_volume_by_borough_df = query_to_dataframe(query_trip_volume_by_borough)
trip_volume_by_borough_df.head()

In [ ]:
filtered_trip_volume_by_borough_df = trip_volume_by_borough_df[(trip_volume_by_borough_df['year'] >= 2020) & (trip_volume_by_borough_df['year'] <= current_year)]
filtered_trip_volume_by_borough_df.info()

In [ ]:
filtered_trip_volume_by_borough_df["trip_date"] = pd.to_datetime(filtered_trip_volume_by_borough_df["trip_date"])

In [ ]:
borough_trip_counts = (
    filtered_trip_volume_by_borough_df.groupby("pickup_borough")["total_trips"]
    .sum()
    .reset_index()
    .sort_values(by="total_trips", ascending=False)
)

fig1 = px.bar(
    borough_trip_counts,
    x="pickup_borough",
    y="total_trips",
    title="Total Trip Volume by Borough",
    labels={"pickup_borough": "Borough", "total_trips": "Total Trips"},
    template="plotly_white",
    text_auto=True
)
fig1.show()

In [ ]:
borough_trend = (
    filtered_trip_volume_by_borough_df.groupby(["trip_date", "pickup_borough"])["total_trips"]
    .sum()
    .reset_index()
)

fig2 = px.line(
    borough_trend,
    x="trip_date",
    y="total_trips",
    color="pickup_borough",
    title="Trip Volume Over Time by Borough",
    labels={"trip_date": "Date", "total_trips": "Total Trips", "pickup_borough": "Borough"},
    template="plotly_white"
)
fig2.show()

### Question 13: How frequently do yellow taxis serve airports (JFK, LaGuardia, ...), and what is the average fare for these trips?


In [ ]:
query_airport_trips_analysis = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.airport_trips_analysis`
"""
airport_trips_analysis_df = query_to_dataframe(query_airport_trips_analysis)
airport_trips_analysis_df.head()

In [ ]:
filtered_airport_trips_analysis_df = airport_trips_analysis_df[(airport_trips_analysis_df['year'] >= 2020) & (airport_trips_analysis_df['year'] <= current_year)]
filtered_airport_trips_analysis_df.info()

In [ ]:
filtered_airport_trips_analysis_df["trip_date"] = pd.to_datetime(filtered_airport_trips_analysis_df["trip_date"])

In [ ]:
airport_trip_counts = (
    filtered_airport_trips_analysis_df.groupby("airport")["total_trips"]
    .sum()
    .reset_index()
    .sort_values(by="total_trips", ascending=False)
)

fig1 = px.bar(
    airport_trip_counts,
    x="airport",
    y="total_trips",
    title="Total Trips by Airport",
    labels={"airport": "Airport", "total_trips": "Total Trips"},
    template="plotly_dark",
    text_auto=True,
    color="total_trips",
    color_continuous_scale="blues"
)
fig1.show()

In [ ]:
airport_trend = (
    filtered_airport_trips_analysis_df.groupby(["trip_date", "airport"])["total_trips"]
    .sum()
    .reset_index()
)

fig2 = px.line(
    airport_trend,
    x="trip_date",
    y="total_trips",
    color="airport",
    title="Airport Trip Trends Over Time",
    labels={"trip_date": "Date", "total_trips": "Total Trips", "airport": "Airport"},
    template="plotly_white",
    line_shape="spline",
    markers=True
)
fig2.show()

In [ ]:
fig3 = px.scatter(
    filtered_airport_trips_analysis_df,
    x="avg_distance",
    y="avg_fare",
    color="airport",
    size="total_trips",
    title="Fare vs Distance for Airport Trips",
    labels={"avg_distance": "Avg Distance (miles)", "avg_fare": "Avg Fare ($)", "airport": "Airport"},
    template="plotly_white",
    hover_data=["total_trips"]
)
fig3.show()

### Question 14: How often do taxis use different rate codes (e.g., standard rate vs. negotiated fares), and how do these rates vary across boroughs?


In [ ]:
query_rate_code_analysis = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.rate_code_analysis`
"""
rate_code_analysis_df = query_to_dataframe(query_rate_code_analysis)
rate_code_analysis_df.head()

In [ ]:
filtered_rate_code_analysis_df = rate_code_analysis_df[(rate_code_analysis_df['year'] >= 2020) & (rate_code_analysis_df['year'] <= current_year)]
filtered_rate_code_analysis_df.info()

In [ ]:
rate_code_counts = (
    filtered_rate_code_analysis_df.groupby("rate_code_description")["total_trips"]
    .sum()
    .reset_index()
    .sort_values(by="total_trips", ascending=False)
)

fig1 = px.bar(
    rate_code_counts,
    x="rate_code_description",
    y="total_trips",
    title="Total Trips by Rate Code",
    labels={"rate_code_description": "Rate Code", "total_trips": "Total Trips"},
    template="plotly_dark",
    text_auto=True,
    color="total_trips",
    color_continuous_scale="oranges"
)
fig1.show()

In [ ]:
borough_rate_distribution = (
    filtered_rate_code_analysis_df.groupby(["pickup_borough", "rate_code_description"])["total_trips"]
    .sum()
    .reset_index()
)

fig2 = px.bar(
    borough_rate_distribution,
    x="pickup_borough",
    y="total_trips",
    color="rate_code_description",
    title="Rate Code Distribution Across Boroughs",
    labels={"pickup_borough": "Borough", "total_trips": "Total Trips", "rate_code_description": "Rate Code"},
    template="plotly_white",
    text_auto=True,
    barmode="stack"
)
fig2.show()

In [ ]:
fig3 = px.box(
    filtered_rate_code_analysis_df,
    x="rate_code_description",
    y="avg_fare",
    title="Fare Distribution by Rate Code",
    labels={"rate_code_description": "Rate Code", "avg_fare": "Average Fare ($)"},
    template="plotly_white",
    color="rate_code_description"
)
fig3.show()

### Question 15: How long do trips typically take, and is there a trend of increasing or decreasing trip durations over time?


In [ ]:
query_trip_duration_analysis = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.trip_duration_analysis`
"""
trip_duration_analysis_df = query_to_dataframe(query_trip_duration_analysis)
trip_duration_analysis_df.head()

In [ ]:
filtered_trip_duration_analysis_df = trip_duration_analysis_df[(trip_duration_analysis_df['year'] >= 2020) & (trip_duration_analysis_df['year'] <= current_year)]
filtered_trip_duration_analysis_df.info()

In [ ]:
fig1 = px.line(
    filtered_trip_duration_analysis_df.groupby(["year", "month"])["avg_trip_duration_min"].mean().reset_index(),
    x="month",
    y="avg_trip_duration_min",
    color="year",
    title="Trend of Average Trip Duration Over Time",
    labels={"month": "Month", "avg_trip_duration_min": "Avg Trip Duration (min)", "year": "Year"},
    template="plotly_dark",
    markers=True
)
fig1.show()

In [ ]:
day_mapping = {0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday", 4: "Friday", 5: "Saturday", 6: "Sunday"}

filtered_trip_duration_analysis_df["day_of_week"] = pd.to_datetime(
    filtered_trip_duration_analysis_df[["year", "month", "day"]]
).dt.dayofweek.map(day_mapping)

fig1 = px.bar(
    filtered_trip_duration_analysis_df.groupby("day_of_week")["avg_trip_duration_min"].mean().reset_index(),
    x="day_of_week",
    y="avg_trip_duration_min",
    title="Average Trip Duration by Day of the Week",
    labels={"day_of_week": "Day of the Week", "avg_trip_duration_min": "Avg Trip Duration (min)"},
    template="plotly_dark",
    color="day_of_week",
    category_orders={"day_of_week": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]}
)
fig1.show()